# 5g Test Data Quality and Robustness

Deliberately corrupt only training data, compare on clean validation data, and stress-test a fixed model with controlled image noise.

**Budget:** 15–25 minutes; a CPU and a few thousand images are enough. No Klique endpoint or token is needed for the default path.

**Before you start:** complete the [JupyterLab introduction](05_Running_Jupyter_Lab.ipynb), prepare Python 3.10+ and the packages in the [support guide](README.md), and select that kernel. Run every cell from a fresh kernel in order. The default is offline: cached real datasets are used when present; otherwise the helper explicitly identifies an 8×8 scikit-learn digits fallback. That fallback is **not MNIST, EMNIST, or Fashion-MNIST**, and its metrics are not comparable to theirs. Downloads are opt-in; read [dataset sources and terms](DATASETS.md) first.

Save your work in the approved persistent workspace. No live Klique launch or restore is established by these local exercises.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
LAB = HERE if (HERE / "image_lab.py").is_file() else HERE / "05_Jupyter"
if not (LAB / "image_lab.py").is_file():
    raise RuntimeError("Start this kernel in the repository root or 05_Jupyter folder.")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))
from image_lab import load_image_data, train_baseline
SEED = 42
OUTPUT = LAB / "lab_outputs"
OUTPUT.mkdir(exist_ok=True)


In [ ]:
from dataclasses import replace
from sklearn.model_selection import train_test_split
import pandas as pd

data = load_image_data("digits", limit_train=1200, limit_test=300, seed=SEED)
fit_idx, val_idx = train_test_split(np.arange(len(data.train_labels)), test_size=0.25,
                                  stratify=data.train_labels, random_state=SEED)
clean = replace(data, train_images=data.train_images[fit_idx], train_labels=data.train_labels[fit_idx],
                test_images=data.train_images[val_idx], test_labels=data.train_labels[val_idx])
rng = np.random.default_rng(SEED)
flipped_labels = clean.train_labels.copy()
flip = rng.choice(len(flipped_labels), size=len(flipped_labels)//4, replace=False)
flipped_labels[flip] = (flipped_labels[flip] + rng.integers(1, 10, size=len(flip))) % 10
masked_images = clean.train_images.copy()
masked_images[:, :, :masked_images.shape[2]//2] = 0
keep = (clean.train_labels != 9) | (rng.random(len(clean.train_labels)) < 0.15)
variants = {
    "clean": clean,
    "25% incorrect training labels": replace(clean, train_labels=flipped_labels),
    "left half missing in training": replace(clean, train_images=masked_images),
    "few training nines": replace(clean, train_images=clean.train_images[keep], train_labels=clean.train_labels[keep]),
}
assert np.array_equal(clean.test_labels, data.train_labels[val_idx])
print("Dataset:", data.name, "Clean validation rows:", len(val_idx))


## Keep a clean comparison boundary

The altered labels and pixels live in copies. Every candidate sees the same clean validation set. Missing-value simulation here is explicit zero-masking, not an arbitrary NaN value the estimator cannot consume. The class-imbalance case makes nines scarce; overall accuracy can hide that damage, so inspect recall for that class.


In [ ]:
from sklearn.metrics import recall_score
rows = []
for name, variant in variants.items():
    fitted, _ = train_baseline(variant, C=1.0, max_iter=100, seed=SEED)
    pred = fitted.predict(clean.test_images.reshape(len(clean.test_images), -1))
    rows.append({"training_variant": name,
                 "validation_accuracy": float(np.mean(pred == clean.test_labels)),
                 "nine_recall": float(recall_score(clean.test_labels == 9, pred == 9, zero_division=0))})
display(pd.DataFrame(rows))


## Stress-test one fixed model, without tuning on the test set

Train the clean, predetermined baseline on all original training examples. Evaluate controlled noise levels on the same held-out test examples. This describes sensitivity to this synthetic perturbation, not a real-world robustness certificate. We do not choose a model using these scores. If you decide to improve robustness next, use new training/validation experiments and reserve a fresh test set.


In [ ]:
fixed_model, _ = train_baseline(data, C=1.0, max_iter=100, seed=SEED)
noise = np.random.default_rng(SEED + 1).normal(size=data.test_images.shape)
robustness = []
for sigma in [0.0, 0.1, 0.25, 0.5]:
    perturbed = np.clip(data.test_images + sigma * noise, 0, 1)
    score = fixed_model.score(perturbed.reshape(len(perturbed), -1), data.test_labels)
    robustness.append({"noise_standard_deviation": sigma, "test_accuracy": float(score)})
robustness = pd.DataFrame(robustness)
display(robustness)
plt.plot(robustness.noise_standard_deviation, robustness.test_accuracy, marker="o")
plt.ylim(0, 1); plt.xlabel("Noise standard deviation"); plt.ylabel("Held-out accuracy")
plt.title("One fixed model under controlled noise"); plt.show()


## Checkpoint, exercise, and cleanup

**Checkpoint:** You have four clean-validation comparisons, class-specific recall, and a separate noise-sensitivity curve for one fixed model.

**Try it:** Add a training-only rotation or a more extreme class imbalance, then measure on the same validation boundary. Explain why random label flips can be harder to detect than a shape mismatch.

**Troubleshooting:** if imports fail, select the kernel where the section requirements were installed. If a data host is blocked, keep downloads off and use the named offline fixture, or ask your administrator to stage the official data in the cache. Do not disable TLS verification. A convergence warning means the solver hit its iteration budget; record it rather than silently claiming an optimum.

Save the notebook and useful artifacts under `lab_outputs/`; it is ignored by Git but may still enter workspace snapshots. Clear outputs before sharing if you used private data. Shut down unused kernels, then stop the managed session and verify its stopped state as in the introduction. Closing a browser tab is not cleanup.

**Next:** [5h: Ask Public Documents](05h_Ask_Public_Documents.ipynb) · **Index:** [Jupyter labs](README.md) · [Orientation](../00_Orientation/00_Orientation.ipynb)
